# Train YOLO Detector

Fine-tunes YOLO on annotated Arctic field images.

**Two-stage training:**
1. **Stage 1** — backbone frozen, only detection head trained (fast convergence)
2. **Stage 2** — full fine-tune from Stage 1 weights (best final performance)

**Input:** `datasets/yolo.zip` — CVAT YOLO 1.1 export
**Output:** `runs/yolo/stage2/weights/best.pt` → copied to `models/yolo_best.pt`

**Classes:** set `KEEP_CLASSES` to the subset you want to train.
Drop `'unsure'` until you have enough annotated examples.

## Cell 1 — Environment

Set your local path. Only edit `BASE_DIR`.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 1 — ENVIRONMENT  ← only edit this cell
# ════════════════════════════════════════════════════════════
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/pollinator-classification')
else:
    BASE_DIR = Path('/Users/lianshi/Downloads/bachelor thesis'
                    '/automated-ecological-image-analysis'
                    '/ml-pipelines/notebooks/pollinator-classification')

MODEL_DIR  = BASE_DIR / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f'Env       : {"Colab" if IN_COLAB else "Local"}')
print(f'BASE_DIR  : {BASE_DIR}  exists={BASE_DIR.exists()}')
print(f'MODEL_DIR : {MODEL_DIR}  exists={MODEL_DIR.exists()}')


## Cell 2 — Training config  ← **edit before training**

- **`KEEP_CLASSES`** — which classes to train. `['fly','butterfly']` is a good start
  if you have limited annotations. Add more classes as annotations grow.
- **`EPOCHS_S1` / `EPOCHS_S2`** — training epochs. More data = more epochs needed.
- **`BATCH`** — reduce if you get out-of-memory errors on Colab (try 16).
- **`COPY_PASTE`** / **`MIXUP`** — augmentation to handle class imbalance.
  These help when some classes have very few annotations.

In [ ]:
import subprocess, shutil, zipfile, json as _json
from pathlib import Path

YOLO_ZIP     = BASE_DIR / 'datasets' / 'yolo.zip'
EXTRACT_TO   = Path('/content/data') if IN_COLAB else BASE_DIR / 'extracted'
OUTPUT_DIR   = (Path('/content/drive/MyDrive/runs/yolo') if IN_COLAB
                else BASE_DIR / 'runs' / 'yolo')

# Classes to train on (drop 'unsure' until more annotations)
CVAT_CLASSES = ['bumblebee','fly','butterfly','other','unsure']
KEEP_CLASSES = ['fly','butterfly']

MODEL_SIZE   = 'yolo11n.pt'
IMG_SIZE     = 640
BATCH        = 32
EPOCHS_S1    = 40   # frozen backbone
EPOCHS_S2    = 70   # full fine-tune
LR_S1        = 1e-3
LR_S2        = 5e-4
SEED         = 42

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Dataset zip : {YOLO_ZIP}  exists={YOLO_ZIP.exists()}')
print(f'Keep classes: {KEEP_CLASSES}')


## Cell 3 — Colab setup

Installs `ultralytics` on Colab. Skipped automatically when running locally.

In [ ]:
if IN_COLAB:
    import subprocess
    subprocess.run(['pip','install','-q','ultralytics'],check=True)
    print('ultralytics installed')


## Cell 4 — Extract + patch dataset

- Extracts `yolo.zip` to a local folder (avoids re-extracting if already done)
- Patches label files to remap CVAT class indices to your `KEEP_CLASSES` order
- Removes annotations for classes not in `KEEP_CLASSES`
- Writes `data.yaml` that YOLO uses to find images and labels

Re-run this cell if you change `KEEP_CLASSES`.

In [ ]:
def extract_zip(zip_path, extract_to, name):
    target=Path(extract_to)/name
    if target.exists() and any(target.iterdir()): print(f'Reusing {target}'); return target
    target.mkdir(parents=True,exist_ok=True)
    local=target.parent/(Path(zip_path).name)
    shutil.copy(str(zip_path),str(local))
    subprocess.run(['unzip','-q','-o',str(local),'-d',str(target)],check=True)
    local.unlink()
    nested=target/name
    if nested.is_dir():
        for item in list(nested.iterdir()): shutil.move(str(item),str(target/item.name))
        nested.rmdir()
    print(f'Extracted: {target}'); return target

def patch_and_write_yaml(root, cvat_classes, keep_classes, config_id):
    root=Path(root); marker=root/'.patched'
    if marker.exists() and marker.read_text()==config_id:
        print('Already patched')
    else:
        ci={n:i for i,n in enumerate(cvat_classes)}
        remap={ci[n]:ni for ni,n in enumerate(keep_classes) if n in ci}
        n_strip=n_rem=0
        for sp in ('train','val','test'):
            ld=root/'labels'/sp
            if not ld.exists(): continue
            for lf in ld.glob('*.txt'):
                orig=lf.read_text().splitlines(); kept=[]
                for line in orig:
                    p=line.strip().split()
                    if not p: continue
                    try: c=int(p[0])
                    except ValueError: continue
                    if c in remap: p[0]=str(remap[c]); kept.append(' '.join(p))
                n_strip+=len(orig)-len(kept)
                if kept: lf.write_text('\n'.join(kept)+'\n')
                else: lf.unlink(); n_rem+=1
        print(f'Patched: stripped {n_strip} lines, removed {n_rem} labels')
        marker.write_text(config_id)
    lines=[f'path: {root}']
    for s in ('train','val','test'):
        if (root/'images'/s).exists(): lines.append(f'{s}: images/{s}')
    lines+=['names:']+[f'  {i}: {n}' for i,n in enumerate(keep_classes)]
    (root/'data.yaml').write_text('\n'.join(lines)+'\n')
    print(f'data.yaml: {keep_classes}')

assert YOLO_ZIP.exists(), f'yolo.zip not found: {YOLO_ZIP}'
config_id=f'keep={"|".join(KEEP_CLASSES)}'
dataset_root=extract_zip(YOLO_ZIP, EXTRACT_TO, 'yolo')
patch_and_write_yaml(dataset_root, CVAT_CLASSES, KEEP_CLASSES, config_id)
YAML_PATH=dataset_root/'data.yaml'
print(f'YAML: {YAML_PATH}')


## Cell 5 — Train

Runs Stage 1 (frozen backbone) then Stage 2 (full fine-tune).
Stage 2 starts from the best Stage 1 checkpoint.
Best weights are automatically copied to `models/yolo_best.pt`.

In [ ]:
from ultralytics import YOLO

# Stage 1: frozen backbone
print('=== Stage 1: frozen backbone ===')
model = YOLO(MODEL_SIZE)
r1 = model.train(
    data=str(YAML_PATH), epochs=EPOCHS_S1, imgsz=IMG_SIZE, batch=BATCH,
    lr0=LR_S1, freeze=10, patience=15, mosaic=1.0, copy_paste=0.3, mixup=0.1,
    cache=True, seed=SEED, project=str(OUTPUT_DIR), name='stage1', exist_ok=True)
s1_best = OUTPUT_DIR/'stage1'/'weights'/'best.pt'
print(f'Stage 1 mAP50: {r1.results_dict.get("metrics/mAP50(B)","n/a")}')

# Stage 2: full fine-tune
print('\n=== Stage 2: full fine-tune ===')
model2 = YOLO(str(s1_best))
r2 = model2.train(
    data=str(YAML_PATH), epochs=EPOCHS_S2, imgsz=IMG_SIZE, batch=BATCH,
    lr0=LR_S2, freeze=0, patience=20, mosaic=1.0, copy_paste=0.3, mixup=0.1,
    cache='disk', seed=SEED, project=str(OUTPUT_DIR), name='stage2', exist_ok=True)
s2_best = OUTPUT_DIR/'stage2'/'weights'/'best.pt'
print(f'Stage 2 mAP50: {r2.results_dict.get("metrics/mAP50(B)","n/a")}')

# Copy to models/
import shutil
dest = MODEL_DIR/'yolo_best.pt'
shutil.copy(str(s2_best), str(dest))
print(f'\nCopied to {dest}')


## Cell 6 — Evaluate

Runs YOLO evaluation on val and test splits.
Prints precision, recall, F1, and AP50 per class.
Use these numbers to compare with crop-based pipelines in `evaluate.ipynb`.

In [ ]:
eval_model = YOLO(str(s2_best))
for split in ('val','test'):
    m = eval_model.val(data=str(YAML_PATH), split=split, imgsz=IMG_SIZE)
    print(f'\n--- {split.upper()} ---')
    for i,cls in enumerate(KEEP_CLASSES):
        try:
            p=m.box.p[i]; r=m.box.r[i]; f1=2*p*r/max(1e-8,p+r)
            print(f'  {cls:12} P={p:.3f}  R={r:.3f}  F1={f1:.3f}  AP50={m.box.ap50[i]:.3f}')
        except (IndexError,AttributeError): print(f'  {cls:12} no detections')
    print(f'  mAP50={m.box.map50:.3f}  mAP50-95={m.box.map:.3f}')
